# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. **`{'A': 2, 'B': 1}`**. Ключ `"A"` встречается дважды, а `.get(c, 0)` позволяет начать счёт с нуля и каждый раз прибавить 1.
2. **`[0, 1, 1]`**. Список передаётся в функцию как один и тот же изменяемый объект: оба вызова `append(1)` меняют исходный `a`.
3. **`2`**. `.lower()` переводит оба варианта Almaty в одну строку `"almaty"`, а множество хранит только уникальные элементы.
4. **`{'ab': 2, 'abc': 3}`**. При повторении ключа `"ab"` словарь перезаписывает его значение, а `len(k)` вычисляет длину строки.
5. **Сначала `5`, затем `None` на новой строке.** Внутри `g(5)` срабатывает `print(5)`, но в функции нет `return`, поэтому переменная `x` получает `None`.


In [1]:
# Проверяем пять предсказаний после записи ответов выше.
print("1:")
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

print("2:")
def f(x):
    x.append(1)
a = [0]
f(a)
f(a)
print(a)

print("3:")
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}
print(len(s))

print("4:")
d = {k: len(k) for k in ["ab", "abc", "ab"]}
print(d)

print("5:")
def g(n):
    if n > 0:
        print(n)
x = g(5)
print(x)


1:
{'A': 2, 'B': 1}
2:
[0, 1, 1]
3:
2
4:
{'ab': 2, 'abc': 3}
5:
5
None


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [3]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [4]:
# Задания 1–4. БАЗА.

# 1. Функция приводит сумму к числу, убирая пробелы между разрядами.
def to_num(s):
    return float(s.strip().replace(" ", ""))

# Одному городу могут соответствовать разные написания.
city_names = {
    "алматы": "Алматы", "almaty": "Алматы",
    "астана": "Астана", "astana": "Астана",
    "шымкент": "Шымкент", "shymkent": "Шымкент",
    "караганда": "Караганда", "karaganda": "Караганда",
    "актобе": "Актобе", "aktobe": "Актобе",
    "атырау": "Атырау", "atyrau": "Атырау",
}

def clean_city(s):
    return city_names[s.strip().lower()]

print("1. Примеры:")
print("to_num('15 990') =", to_num("15 990"))
print("clean_city(' Almaty ') =", clean_city(" Almaty "))
print("Уникальных городов после очистки:", len({clean_city(r["city"]) for r in rows}))

# 2. Считаем все заказы по категориям, включая отменённые.
category_counts = {}
for r in rows:
    category = r["category"]
    category_counts[category] = category_counts.get(category, 0) + 1

print("\n2. Количество заказов по категориям:")
for category, count in sorted(category_counts.items(), key=lambda item: item[1], reverse=True):
    print(f"{category:<15} {count:>5}")

# 3. Учитываем только оплаченные заказы.
category_revenue = {}
category_paid_counts = {}
for r in rows:
    if r["status"] == "оплачен":
        category = r["category"]
        amount = to_num(r["amount"])
        category_revenue[category] = category_revenue.get(category, 0) + amount
        category_paid_counts[category] = category_paid_counts.get(category, 0) + 1

print("\n3. Выручка и средний чек оплаченных заказов:")
print(f"{'Категория':<15} {'Оплачено':>10} {'Выручка, ₸':>17} {'Средний чек, ₸':>18}")
for category, revenue in sorted(category_revenue.items(), key=lambda item: item[1], reverse=True):
    paid_count = category_paid_counts[category]
    average = revenue / paid_count
    print(f"{category:<15} {paid_count:>10} {revenue:>17,.0f} {average:>18,.2f}")

# 4. Подсчитываем количество всех заказов отдельно для каждого месяца.
monthly_counts = {}
for r in rows:
    month = r["order_date"][:7]
    monthly_counts[month] = monthly_counts.get(month, 0) + 1

print("\n4. Количество заказов по месяцам:")
for month in sorted(monthly_counts):
    print(f"{month}: {monthly_counts[month]} заказов")


1. Примеры:
to_num('15 990') = 15990.0
clean_city(' Almaty ') = Алматы
Уникальных городов после очистки: 6

2. Количество заказов по категориям:
Одежда           1529
Электроника      1185
Дом              1105
Красота           991
Книги             653

3. Выручка и средний чек оплаченных заказов:
Категория         Оплачено        Выручка, ₸     Средний чек, ₸
Электроника           1065        65,270,240          61,286.61
Одежда                1379        18,321,080          13,285.77
Дом                    987        10,516,320          10,654.83
Красота                890         5,507,620           6,188.34
Книги                  583         2,294,100           3,934.99

4. Количество заказов по месяцам:
2026-09: 2782 заказов
2026-10: 2681 заказов


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [5]:
# Задания 5–8. СТАНДАРТ.

# 5. Выручка оплаченных заказов отдельно за сентябрь и октябрь.
monthly_revenue = {}
for r in rows:
    if r["status"] == "оплачен":
        month = r["order_date"][:7]
        monthly_revenue[month] = monthly_revenue.get(month, 0) + to_num(r["amount"])

september = monthly_revenue["2026-09"]
october = monthly_revenue["2026-10"]
original_change = (october - september) / september * 100

print("5. Выручка по месяцам:")
for month in sorted(monthly_revenue):
    print(f"{month}: {monthly_revenue[month]:,.0f} ₸")
print(f"Изменение октября к сентябрю: {original_change:+.2f}%")

# 6. Отменённые заказы / все заказы по каждому нормализованному городу.
city_counts = {}
city_cancelled = {}
for r in rows:
    city = clean_city(r["city"])
    city_counts[city] = city_counts.get(city, 0) + 1
    if r["status"] == "отменён":
        city_cancelled[city] = city_cancelled.get(city, 0) + 1

print("\n6. Доля отменённых заказов по городам:")
for city in sorted(city_counts):
    cancelled = city_cancelled.get(city, 0)
    share = cancelled / city_counts[city] * 100
    print(f"{city:<12} {cancelled:>3}/{city_counts[city]:<4} = {share:.1f}%")

# 7. Накапливаем выручку оплаченных заказов по ID клиента.
customer_revenue = {}
for r in rows:
    if r["status"] == "оплачен":
        customer = r["customer_id"]
        customer_revenue[customer] = customer_revenue.get(customer, 0) + to_num(r["amount"])

top_five = sorted(customer_revenue.items(), key=lambda item: item[1], reverse=True)[:5]
print("\n7. Топ-5 клиентов по выручке:")
for place, (customer, revenue) in enumerate(top_five, start=1):
    print(f"{place}. {customer:<10} {revenue:>12,.0f} ₸")

# 8. Множество оставляет только уникальные customer_id в каждом городе.
city_customers = {}
for r in rows:
    city = clean_city(r["city"])
    city_customers.setdefault(city, set()).add(r["customer_id"])

print("\n8. Уникальные клиенты по городам:")
for city in sorted(city_customers):
    print(f"{city:<12} {len(city_customers[city]):>5}")


5. Выручка по месяцам:
2026-09: 57,489,850 ₸
2026-10: 44,419,510 ₸
Изменение октября к сентябрю: -22.74%

6. Доля отменённых заказов по городам:
Актобе        41/397  = 10.3%
Алматы       172/2188 = 7.9%
Астана       123/1390 = 8.8%
Атырау        11/146  = 7.5%
Караганда     48/538  = 8.9%
Шымкент      164/804  = 20.4%

7. Топ-5 клиентов по выручке:
1. TEST         29,999,970 ₸
2. C01627          257,180 ₸
3. C00485          215,920 ₸
4. C01338          192,080 ₸
5. C02122          191,670 ₸

8. Уникальные клиенты по городам:
Актобе         358
Алматы        1351
Астана         981
Атырау         138
Караганда      451
Шымкент        655


**Ответ на задачу 7.** На первом месте клиент **`TEST`** с выручкой **29 999 970 ₸**. Это не обычный покупатель, а тестовый идентификатор: у него три оплаченных заказа по **9 999 990 ₸** каждый. Эти искусственно большие суммы искажают показатели и рейтинг клиентов, поэтому для реального анализа продаж его нужно исключить.

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [6]:
# Задания 9–11. ЧЕЛЛЕНДЖ.

# 9. Проверяем идентичность ВСЕХ столбцов: tuple можно хранить в set.
seen = set()
unique_rows = []
duplicate_rows = []
for r in rows:
    row_key = tuple(r.values())
    if row_key in seen:
        duplicate_rows.append(r)
    else:
        seen.add(row_key)
        unique_rows.append(r)

duplicates_by_month = {}
for r in duplicate_rows:
    month = r["order_date"][:7]
    duplicates_by_month[month] = duplicates_by_month.get(month, 0) + 1

print("9. Полных дублей:", len(duplicate_rows))
print("Распределение дублей по месяцам:", duplicates_by_month)
print("Строк осталось после удаления дублей:", len(unique_rows))

# 10. Убираем дубли и тестового клиента из выручки обоих месяцев.
clean_monthly_revenue = {}
for r in unique_rows:
    if r["status"] == "оплачен" and r["customer_id"] != "TEST":
        month = r["order_date"][:7]
        clean_monthly_revenue[month] = clean_monthly_revenue.get(month, 0) + to_num(r["amount"])

clean_september = clean_monthly_revenue["2026-09"]
clean_october = clean_monthly_revenue["2026-10"]
corrected_change = (clean_october - clean_september) / clean_september * 100

print("\n10. Выручка после очистки:")
for month in sorted(clean_monthly_revenue):
    print(f"{month}: {clean_monthly_revenue[month]:,.0f} ₸")
print(f"До очистки: {original_change:+.2f}%")
print(f"После очистки: {corrected_change:+.2f}%")

# 11. Число заказов каждого клиента без полных дублей.
customer_counts = {}
for r in unique_rows:
    customer = r["customer_id"]
    customer_counts[customer] = customer_counts.get(customer, 0) + 1

most_orders_customer = max(customer_counts, key=customer_counts.get)
print("\n11. Больше всего заказов у клиента:",
      most_orders_customer, "—", customer_counts[most_orders_customer], "заказов")


9. Полных дублей: 180
Распределение дублей по месяцам: {'2026-09': 180}
Строк осталось после удаления дублей: 5283

10. Выручка после очистки:
2026-09: 35,243,410 ₸
2026-10: 34,419,520 ₸
До очистки: -22.74%
После очистки: -2.34%

11. Больше всего заказов у клиента: C01818 — 10 заказов


### 12. Сообщение менеджеру

По исходным данным выручка октября по сравнению с сентябрём упала на **22,7%**, однако эта оценка оказалась сильно искажена. Мы обнаружили **180 полных дублей заказов в сентябре** и тестового клиента **`TEST`** с тремя фиктивно крупными покупками. После исключения дублей и клиента `TEST` выручка составила **35 243 410 ₸ в сентябре** и **34 419 520 ₸ в октябре**, то есть реальное снижение — около **2,3%**. Для отчётности следует пользоваться очищенными данными и проверять повторяющиеся строки и тестовые заказы перед сравнением месяцев.